In [7]:
#Data Collection
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [8]:
from google.colab import drive
auth.authenticate_user()
from google.colab import files
drive.mount('/content/drive')

NameError: name 'auth' is not defined

In [ ]:
#Loading the data set
data = gutenberg.raw('shakespeare-hamlet.txt')
#Save the data to a text file
with open('hamlet.txt', 'w') as f:
    f.write(data)
#Download the file to local machine
files.download('hamlet.txt')

In [ ]:
#Data Preprocessing
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

#load the data set
try:
    with open('hamlet.txt', 'r') as f:
        text = f.read().lower()
except FileNotFoundError:
    # If file doesn't exist, load from gutenberg directly
    from nltk.corpus import gutenberg
    text = gutenberg.raw('shakespeare-hamlet.txt').lower()

print(f"Text length: {len(text)}")
print(f"First 200 characters: {text[:200]}")

#tokenize the text create a tokenizer object and fit it on the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print(f"Total words: {total_words}")
total_words

In [ ]:
tokenizer.word_index

In [ ]:
text

In [ ]:
#Create Input sequences
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Number of input sequences: {len(input_sequences)}")
if input_sequences:
    print(f"First sequence: {input_sequences[0]}")
    print(f"Max sequence length: {max(len(seq) for seq in input_sequences)}")

In [ ]:
input_sequences[:10]

In [ ]:
#Apply padding to the input sequences
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
input_sequences[:10]
print(f"Shape of max sequences: {max_sequence_len}")

In [ ]:
#Create predictors and label
predictors, label = input_sequences[:,:-1], input_sequences[:,-1]
label = tf.keras.utils.to_categorical(label, num_classes=total_words)

In [ ]:
label

In [ ]:
#Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(predictors, label, test_size=0.2, random_state=42)
X_train

In [ ]:
#Training the model using LSTM RNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
#Define the model architecture
from tensorflow.keras.layers import Input

model = Sequential()
model.add(Input(shape=(max_sequence_len-1,)))
model.add(Embedding(total_words, 300))
model.add(LSTM(256, return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(128))
model.add(Dense(total_words, activation='softmax'))

#GRU model
#model.add(Input(shape=(max_sequence_len-1,)))
#model.add(Embedding(total_words, 300))
#model.add(GRU(256, return_sequences=True))
#model.add(Dropout(0.3))
#model.add(GRU(128))
#model.add(Dense(total_words, activation='softmax'))

In [ ]:
#Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
#Train the model
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=15)

print(f"Training data shape: X_train {X_train.shape}, y_train {y_train.shape}")
if len(X_train) == 0:
    print("Error: No training data available. Please check data loading and preprocessing.")
else:
    history = model.fit(X_train, y_train, epochs=300, 
                        validation_data=(X_test, y_test), callbacks=[early_stop])

In [ ]:
#Function to predict the next word in a sequence
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [ ]:
input_text = "Tis not alone my Inky"
max_sequence_len = model.input_shape[1]+1
next_word=predict_next_word(model, tokenizer, input_text.lower(), max_sequence_len)
print(f"Next predicted word : {next_word}")

In [ ]:
def predict_next(input_text, model, tokenizer, max_sequence_len, top_k=10):
    seq = tokenizer.texts_to_sequences([input_text.lower()])[0]
    seq = pad_sequences([seq], maxlen=max_sequence_len-1, padding='pre')
    preds = model.predict(seq)[0]
    top_indices = preds.argsort()[-top_k:][::-1]
    return [(i, tokenizer.index_word.get(i, '<UNK>'), float(preds[i])) for i in top_indices]

input_text = "Tis not alone my Inky"
results = predict_next(input_text, model, tokenizer, max_sequence_len, top_k=20)
print("Top predictions (index, word, prob):")
for idx, word, prob in results:
    print(idx, word, f"{prob:.4f}")

In [ ]:
#Save model
model.save("next_word_prediction_lstm.keras")
from google.colab import files
files.download("next_word_prediction_lstm.keras")

In [ ]:
#Save the tockenizer
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
    files.download('tokenizer.pickle')